# AI-Finance: View Training Results

Run `python services/python/scripts/run_training.py` first, then use this notebook to inspect results.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

RESULTS_DIR = Path("../models/results")

print("=" * 65)
print(f"{'Model':<15} {'AUC':>8} {'Acc':>8} {'Precision':>11} {'Signals':>9}")
print("-" * 55)
for name in ["xgboost", "lightgbm", "lstm", "ensemble"]:
    p = RESULTS_DIR / f"{name}_metrics.json"
    if not p.exists():
        continue
    m = json.loads(p.read_text())
    print(f"{m['model']:<15} {m.get('auc',0):>7.3f} {m.get('accuracy',0):>7.3f} "
          f"{m.get('precision',0):>10.3f} {m.get('signals',0):>9}")
print("=" * 65)

In [ ]:
# Feature importance from saved XGBoost model
import joblib

MODEL_DIR = Path("../models")
xgb = joblib.load(MODEL_DIR / "xgboost_latest.joblib")
features = (MODEL_DIR / "feature_cols.txt").read_text().strip().split("\n")

importance = pd.Series(xgb.feature_importances_, index=features).sort_values(ascending=False)
print("Top 15 Features:")
for feat, imp in importance.head(15).items():
    bar = "#" * int(imp * 100)
    print(f"  {feat:<25} {imp:.4f} {bar}")

In [ ]:
# Ensemble probability distribution
ens = np.load(RESULTS_DIR / "ensemble_predictions.npz")
probs = ens["probs"]
actuals = ens["actuals"]

print(f"Total predictions: {len(probs):,}")
print(f"Prob stats: mean={probs.mean():.3f}, std={probs.std():.3f}")
print(f"\nBy confidence bucket:")
for lo, hi in [(0.5, 0.55), (0.55, 0.6), (0.6, 0.65), (0.65, 0.7), (0.7, 1.0)]:
    mask = (probs >= lo) & (probs < hi)
    if mask.sum() > 0:
        precision = actuals[mask].mean()
        print(f"  [{lo:.2f}-{hi:.2f}): n={mask.sum():>6}, precision={precision:.3f}")